# FINAL PROJECT

**Deliverables**:
  1. Your version of this notebook, where you create the required agents (complete all the associated tasks)

  2. After training your agents, you might want to save those learnt values. This can be a dictionary with the `optimal policy`, or `Q-values`. If an approximate method was used, you can provide the `final weights`, etc.

  3. A short report (no more than 10 pages), where you explain your methodology, chosen parameters, visualisations of the learning process (how rewards increase over time, or how loss functions decrease).

  4. Upload to `Faser` a .zip file of all the deliverables (including th optimal Q-values/weights/policies)

**Evaluation**: Your projects will be evaluated primarily on the development of the agents, not necessarily in their performance. Being said that, better performance will lead in general to better marks.

## 0. Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import gymnasium as gym
from IPython.display import Image, display
import imageio

from collections import defaultdict
from tqdm import tqdm
from optuna.trial import FixedTrial

from src.logger import LogManager
from src.utils import (
    save_policy, load_policy, plot_learning_curve, 
    plot_smoothed_learning_curve, q_stats
)
from src.optimization import param_opt_pipeline, get_params

In [ ]:
manager = LogManager()

main_log = manager.get_logger("Main", "main.log")

In [ ]:
env = gym.make("Acrobot-v1", render_mode = "rgb_array")
state = env.reset()

main_log.info("--- New Session Started ---")
main_log.info(f"Environment: Acrobot-v1")
main_log.info(f"Initial State: {state}")
main_log.info(f"State Space: {env.observation_space}")
main_log.info(f"Action Space: {env.action_space}")

[ACROBOT ENVIRONMENT](https://gymnasium.farama.org/environments/classic_control/acrobot/)

Below is the default discretisation (for the states. Actions are already discrete). If you consider that a different one, could lead to better performance, feel free to modify it, and indicate this in your report.

In [ ]:
ANGLE_BINS = 10
VEL_BINS = 12

LOW = np.array([-np.pi, -np.pi, -6, -10])
HIGH = np.array([ np.pi,  np.pi,  6,  10])
BINS = np.array([ANGLE_BINS, ANGLE_BINS, VEL_BINS, VEL_BINS])

def transform_state(obs):
    cos1, sin1, cos2, sin2, w1, w2 = obs
    
    theta1 = np.arctan2(sin1, cos1)
    theta2 = np.arctan2(sin2, cos2)
    
    return np.array([theta1, theta2, w1, w2])

def discretise(obs):
    state = transform_state(obs)
    
    ratios = (state - LOW) / (HIGH - LOW)
    ratios = np.clip(ratios, 0, 1)
    
    discrete = (ratios * (BINS - 1)).astype(int)
    return tuple(discrete)

In [ ]:
main_log.info(f"Discretization Bins: {BINS}")

## 1. Notebook Utilities

In [ ]:
def select_action(policy, state, mode="greedy", epsilon=0.05, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    values = policy[state]
    n_actions = len(values)

    # Detect policy type by whether values sum to ~1 (probability dist) or not (Q-values)
    is_prob_policy = np.isclose(values.sum(), 1.0)

    if mode == "greedy":
        return int(np.argmax(values))

    elif mode == "stochastic":
        if is_prob_policy:
            return int(rng.choice(n_actions, p=values))
        else:
            # Softmax over Q-values as a fallback
            exp_q = np.exp(values - np.max(values))
            probs = exp_q / exp_q.sum()
            return int(rng.choice(n_actions, p=probs))

    elif mode == "epsilon_greedy":
        if rng.random() < epsilon:
            return int(rng.integers(n_actions))
        return int(np.argmax(values))

In [ ]:
def evaluate_policy(policy, n_episodes=20, mode="greedy", seed=42):
    env = gym.make("Acrobot-v1")

    rewards = []
    steps = []

    for ep in range(n_episodes):
        rng = np.random.default_rng(seed + ep)

        obs, _ = env.reset()
        state = discretise(obs)

        done = False
        total_reward = 0
        t = 0

        while not done:
            action = select_action(policy, state, mode = mode, rng = rng)
            obs, reward, term, trunc, _ = env.step(action)

            state = discretise(obs)
            total_reward += reward
            t += 1
            done = term or trunc

        rewards.append(total_reward)
        steps.append(t)

    return {
        "mean_reward": np.mean(rewards),
        "std_reward": np.std(rewards),
        "mean_steps": np.mean(steps),
        "success_rate": np.mean(np.array(steps) < 500)
    }

In [ ]:
def record_episode(policy, filename, mode="greedy"):
    env = gym.make("Acrobot-v1", render_mode="rgb_array")
    frames = []

    obs, _ = env.reset()
    state = discretise(obs)

    done = False

    while not done:
        frames.append(env.render())
        action = select_action(policy, state, mode=mode)
        obs, reward, term, trunc, _ = env.step(action)

        state = discretise(obs)
        done = term or trunc

    imageio.mimsave(filename, frames, fps=50)

In [ ]:
def run_experiment(algorithm_func, params):
    name = algorithm_func.__name__
    main_log.info(f"Starting Experiment: {name}")
    
    params["seed"] = 42
    params["n_episodes"] = 2000

    main_log.info(f"Hyperparameters: {params}")

    # ---------------- TRAIN ----------------
    model, rewards = algorithm_func(env, params=params)

    main_log.info(f"{name} Training Completed")

    # ---------------- SAVE ----------------
    save_policy(model, name)

    # ---------------- EVALUATE ----------------
    eval_greedy = evaluate_policy(model, mode="greedy")
    eval_stochastic = evaluate_policy(model, mode="stochastic")

    main_log.info(f"{name} Evaluation (Greedy): {eval_greedy}")

    print("\n=== EVALUATION ===")
    print("Greedy:", eval_greedy)
    print("Stochastic:", eval_stochastic)

    # ---------------- PLOTS ----------------
    plot_learning_curve(rewards, name)
    plot_smoothed_learning_curve(rewards, name)

    # ---------------- RECORD ----------------
    record_episode(model, f"results/{name}_greedy.gif", mode="greedy")
    record_episode(model, f"results/{name}_stochastic.gif", mode="stochastic")

    return model, rewards, {
        "greedy": eval_greedy,
        "stochastic": eval_stochastic
    }

## 2. Algorithms

### SARSA

In [ ]:
def alg_SARSA(env, params):
    """
    SARSA implementation.
    """

    # Configuration parameters
    seed = params["seed"]
    rng = np.random.default_rng(seed)
    
    # Problem-specific parameters
    n_episodes = params["n_episodes"]
    n_actions = env.action_space.n
    
    gamma = params["gamma"]
    epsilon = params["epsilon"]
    epsilon_min = params["epsilon_min"]
    
    # Algorithm-specific parameters
    alpha = params["alpha"]
    epsilon_decay = params["epsilon_decay"]

    # --- Initialization ---
    
    Q = defaultdict(lambda: np.zeros(n_actions))
    rewards_history = []

    for ep in tqdm(range(n_episodes), desc = "Training SARSA"):
        obs, _ = env.reset()
        state = discretise(obs)
        action = select_action(
            policy = Q, 
            state = state, 
            mode = "epsilon_greedy", 
            epsilon = epsilon, 
            rng = rng
        )

        total_reward = 0
        done = False

        while not done:
            next_obs, reward, term, trunc, _ = env.step(action)
            next_state = discretise(next_obs)
            
            next_action = select_action(
                policy = Q, 
                state = next_state, 
                mode = "epsilon_greedy", 
                epsilon = epsilon, 
                rng = rng
            )

            Q[state][action] += alpha * (reward + gamma * Q[next_state][next_action] - Q[state][action])

            state, action = next_state, next_action
            total_reward += reward
            done = term or trunc

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_reward)

    return Q, rewards_history

### Q-Learning [UG: 30 /  PGT: 20] marks


In here you will need to implement a version of Q-Learning, test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of Q-learning
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [ ]:
def alg_Q(env, params: dict):
    """
    Q-Learning implementation using a centralized parameter dictionary.
    """

    # Configuration parameters
    seed = params["seed"]
    rng = np.random.default_rng(seed)
    show_progress = params["show_progress"]
    
    # Problem-specific parameters
    n_episodes = params["n_episodes"]
    n_actions = env.action_space.n
    
    gamma = params["gamma"]
    epsilon = params["epsilon"]
    epsilon_min = params["epsilon_min"]
    
    # Algorithm-specific parameters
    alpha = params["alpha"]
    epsilon_decay = params["epsilon_decay"]

    # --- Initialization ---
    
    Q = defaultdict(lambda: np.zeros(n_actions))
    rewards_history = []

    iterator = range(n_episodes)
    if show_progress:
        iterator = tqdm(iterator, desc="Training Q-Learning", leave = False)

    for ep in iterator:
        obs, _ = env.reset()
        state = discretise(obs)

        total_reward = 0
        done = False

        while not done:
            action = select_action(
                policy = Q,
                state = state,
                mode = "epsilon_greedy",
                epsilon = epsilon,
                rng = rng
            )

            next_obs, reward, term, trunc, _ = env.step(action)
            next_state = discretise(next_obs)

            # Off policy
        
            Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])

            state = next_state
            total_reward += reward
            done = term or trunc

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_reward)

    return Q, rewards_history

### n-step SARSA [UG: 35 / PGT: 30] marks


In here you will need to implement a version of n-step SARSA, test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of n-step SARSA
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [ ]:
def alg_nStep_SARSA(env, params):
    """
    n-Step SARSA implementation using a centralized parameter dictionary.
    """

    # Configuration parameters
    seed = params["seed"]
    rng = np.random.default_rng(seed)
    show_progress = params["show_progress"]
    
    # Problem-specific parameters
    n_episodes = params["n_episodes"]
    n_actions = env.action_space.n
    
    gamma = params["gamma"]
    epsilon = params["epsilon"]
    epsilon_min = params["epsilon_min"]
    
    # Algorithm-specific parameters
    alpha = params["alpha"]
    epsilon_decay = params["epsilon_decay"]
    n = params["n"]

    # --- Initialization ---
    
    Q = defaultdict(lambda: np.zeros(n_actions))
    # Initialize pi to be epsilon-greedy with respect to Q,
    rewards_history = []

    iterator = range(n_episodes)
    if show_progress:
        iterator = tqdm(iterator, desc="Training n-Step SARSA", leave = False)

    for ep in iterator:

        obs, _ = env.reset()
        state = discretise(obs)
        action = select_action(
            policy = Q,
            state = state,
            mode = "epsilon_greedy",
            epsilon = epsilon,
            rng = rng
        )

        states = [state]
        actions = [action]
        rewards = [0.0]

        total_reward = 0

        T = np.inf
        tau = 0
        t = 0

        while tau < T - 1:
            if t < T:
                # Take action A_t, observe R_{t+1}, S_{t+1}
                next_obs, reward, term, trunc, _ = env.step(actions[t])
                next_state = discretise(next_obs)
                
                rewards.append(reward)
                states.append(next_state)

                total_reward += reward

                if term or trunc:
                    T = t + 1
                else:
                    # Select and store next action A_{t+1}
                    next_action = select_action(
                        policy = Q,
                        state = next_state,
                        mode = "epsilon_greedy",
                        epsilon = epsilon,
                        rng = rng
                    )
                    actions.append(next_action)
            
            # tau is the time whose estimate is being updated
            tau = t - n + 1

            if tau >= 0:
                # Calculate the n-step return G
                # G = sum_{i=tau+1}^{min(tau+n, T)} gamma^(i-tau-1) * R_i
                G = 0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                
                # Add the bootstrapped value of the state at the end of the window
                if tau + n < T:
                    S_n = states[tau + n]
                    A_n = actions[tau + n]
                    G += (gamma ** n) * Q[S_n][A_n]
                
                S_tau = states[tau]
                A_tau = actions[tau]
                Q[S_tau][A_tau] += alpha * (G - Q[S_tau][A_tau])

            t += 1

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_reward)

    return Q, rewards_history

### REINFORCE [UG: 35 / PGT: 30 ] marks


In here you will need to implement a version of REINFORCE, test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of REINFORCE
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [ ]:
def get_probs(h):
    exp_h = np.exp(h - np.max(h)) # Stability
    return exp_h / np.sum(exp_h)

In [ ]:
def alg_REINFORCE_B(env, params):
    """
    REINFORCE with baseline (state-value function).
    """

    seed = params["seed"]
    rng = np.random.default_rng(seed)
    show_progress = params["show_progress"]

    n_episodes = params["n_episodes"]
    n_actions = env.action_space.n
    gamma = params["gamma"]

    alpha_theta = params["alpha_theta"]
    alpha_w = params["alpha_w"]

    # Policy (logits) and baseline
    h = defaultdict(lambda: np.zeros(n_actions))
    v = defaultdict(float)

    rewards_history = []

    iterator = range(n_episodes)
    if show_progress:
        iterator = tqdm(iterator, desc="Training REINFORCE-B", leave=False)

    for ep in iterator:
        obs, _ = env.reset()
        state = discretise(obs)

        ep_states = []
        ep_actions = []
        ep_rewards = []

        done = False

        # --- Generate trajectory ---
        while not done:
            probs = get_probs(h[state])
            action = rng.choice(n_actions, p=probs)

            ep_states.append(state)
            ep_actions.append(action)

            obs, reward, term, trunc, _ = env.step(action)
            state = discretise(obs)

            ep_rewards.append(reward)
            done = term or trunc

        rewards_history.append(np.sum(ep_rewards))

        # --- Compute returns ---
        T = len(ep_rewards)
        returns = np.zeros(T)

        G = 0.0
        for t in reversed(range(T)):
            G = ep_rewards[t] + gamma * G
            returns[t] = G

        # Normalize returns - only when there's meaningful variance
        if returns.std() > 1e-3:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        else:
            returns = returns - returns.mean()  # just center, don't scale

        # --- Update ---
        for t in range(T):
            s = ep_states[t]
            a = ep_actions[t]
            G_t = returns[t]

            # Baseline update
            delta = G_t - v[s]
            v[s] += alpha_w * delta

            # Policy gradient
            probs = get_probs(h[s])
            grad = -probs
            grad[a] += 1.0

            h[s] += alpha_theta * (gamma ** t) * delta * grad

    # Convert logits → policy
    n_act = n_actions  # capture for lambda
    policy = defaultdict(lambda: np.full(n_act, 1.0 / n_act))
    policy.update({s: get_probs(h_s) for s, h_s in h.items()})
    
    return policy, rewards_history

### SARSA($\lambda$) [PGT: 20 marks ]


In here you will need to implement a version of SARSA($\lambda$), test and train it, finding the best set of parameters that allows the method to find the solution.

You will need to include:

- implementation of SARSA($\lambda$)
- optimal set of parameters (after trying different combinations)
- visualisations (e.g., execution and/or evolution of rewards)
- Optimal policy and/or Optimal values of Q (e.g., saved as dictionaries )

In [ ]:
def alg_SARSA_Lambda(env, params):
    """
    True Online SARSA(λ) — Sutton & Barto, Chapter 12.
    Tabular adaptation: Q-table as linear function approximation
    with implicit one-hot features x(s,a).
    """

    seed = params["seed"]
    rng = np.random.default_rng(seed)
    show_progress = params["show_progress"]

    n_episodes = params["n_episodes"]
    n_actions = env.action_space.n

    gamma = params["gamma"]
    epsilon = params["epsilon"]
    epsilon_min = params["epsilon_min"]

    alpha = params["alpha"]
    epsilon_decay = params["epsilon_decay"]
    lam = params["lambda"]

    # w^T x(s,a) ≈ Q(s,a) — equivalent to a Q-table under one-hot features
    Q = defaultdict(lambda: np.zeros(n_actions))
    rewards_history = []

    iterator = range(n_episodes)
    if show_progress:
        iterator = tqdm(iterator, desc="Training True Online SARSA(λ)", leave=False)

    for ep in iterator:
        obs, _ = env.reset()
        state = discretise(obs)

        # Eligibility trace table — dutch traces, one scalar per (s,a)
        E = defaultdict(lambda: np.zeros(n_actions))

        action = select_action(Q, state, mode="epsilon_greedy", epsilon=epsilon, rng=rng)

        Q_old = 0.0  # Q_old ← 0 as per pseudocode
        total_reward = 0.0
        done = False

        while not done:
            next_obs, reward, term, trunc, _ = env.step(action)
            next_state = discretise(next_obs)

            next_action = select_action(Q, next_state, mode="epsilon_greedy", epsilon=epsilon, rng=rng)

            Q_cur  = Q[state][action]                                          # Q  ← w^T x
            Q_next = 0.0 if (term or trunc) else Q[next_state][next_action]   # Q' ← w^T x'

            # δ ← R + γQ' - Q
            delta = reward + gamma * Q_next - Q_cur

            # Dutch trace: z ← γλz + (1 - αγλ z^T x) x
            # Under one-hot features, z^T x = E[state][action], x contributes only at (state, action)
            E[state][action] = (
                E[state][action] * (gamma * lam)
                + (1.0 - alpha * gamma * lam * E[state][action])
            )

            # True Online weight update:
            # w ← w + α(δ + Q - Q_old)z - α(Q - Q_old)x
            # Under one-hot x, the last term only touches (state, action)
            correction = Q_cur - Q_old

            for s in list(E.keys()):
                Q[s] += alpha * (delta + correction) * E[s]
                E[s] *= gamma * lam

                if np.all(np.abs(E[s]) < 1e-6):
                    del E[s]

            # Undo the double-update on x: subtract α(Q - Q_old) at (state, action)
            Q[state][action] -= alpha * correction

            Q_old = Q_next
            state, action = next_state, next_action
            total_reward += reward
            done = term or trunc

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_reward)

    return Q, rewards_history

## 3. Experiments

### Smoketest

In [ ]:
# ============================================================
# SMOKE TEST — run this before the full optimization
# Verifies the entire pipeline in ~30 seconds
# Comment out or delete before final submission
# ============================================================

def smoke_test():
    SMOKE_PARAMS_TD = {
        "n_episodes": 5,
        "gamma": 0.99,
        "epsilon": 1.0,
        "epsilon_min": 0.05,
        "epsilon_decay": 0.99,
        "alpha": 0.15,
        "seed": 42,
        "show_progress": False,
    }

    SMOKE_PARAMS_NSTEP = {**SMOKE_PARAMS_TD, "n": 3}
    SMOKE_PARAMS_REINFORCE = {
        "n_episodes": 5,
        "gamma": 0.99,
        "alpha_theta": 1e-3,
        "alpha_w": 1e-2,
        "seed": 42,
        "show_progress": False,
    }
    SMOKE_PARAMS_SARSA_L = {**SMOKE_PARAMS_TD, "lambda": 0.8}

    tests = [
        (alg_SARSA,        "alg_SARSA",        SMOKE_PARAMS_TD),
        (alg_Q,            "alg_Q",            SMOKE_PARAMS_TD),
        (alg_nStep_SARSA,  "alg_nStep_SARSA",  SMOKE_PARAMS_NSTEP),
        (alg_REINFORCE_B,  "alg_REINFORCE_B",  SMOKE_PARAMS_REINFORCE),
        (alg_SARSA_Lambda, "alg_SARSA_Lambda",  SMOKE_PARAMS_SARSA_L),
    ]

    smoke_env = gym.make("Acrobot-v1")

    for func, name, params in tests:
        try:
            model, rewards = func(smoke_env, params=params)

            # Check outputs are well-formed
            assert len(rewards) == params["n_episodes"], "Wrong number of episodes"
            assert all(isinstance(r, float) for r in rewards), "Rewards should be floats"
            assert len(model) > 0, "Model is empty"

            # Check select_action works on a real state
            obs, _ = smoke_env.reset()
            state = discretise(obs)
            a_greedy     = select_action(model, state, mode="greedy")
            a_eps        = select_action(model, state, mode="epsilon_greedy", epsilon=0.5)
            a_stochastic = select_action(model, state, mode="stochastic")
            assert a_greedy     in range(3), f"Bad greedy action: {a_greedy}"
            assert a_eps        in range(3), f"Bad epsilon-greedy action: {a_eps}"
            assert a_stochastic in range(3), f"Bad stochastic action: {a_stochastic}"

            # Check save/load roundtrip
            save_policy(model, f"_smoke_{name}")
            loaded = load_policy(f"_smoke_{name}")
            a_loaded = select_action(loaded, state, mode="greedy")
            assert a_loaded in range(3), "Bad action after load"

            # Check evaluate_policy runs
            eval_result = evaluate_policy(model, n_episodes=2, mode="greedy")
            assert "mean_reward" in eval_result, "Missing mean_reward in eval"

            print(f"  ✓ {name}")

        except Exception as e:
            print(f"  ✗ {name}: {e}")
            raise  # re-raise so you see the full traceback

    # Cleanup smoke model files
    import glob
    for f in glob.glob("models/_smoke_*.pkl"):
        os.remove(f)

    print("\nAll smoke tests passed. Safe to run full pipeline.")

In [ ]:
smoke_test()

### Baseline

In [ ]:
# --- HYPERPARAMETERS ---
# These parameters are SARSA / Q-learning specific. 

SARSA_PARAMS = {
    # Problem parameters: <- Fixed parameters to the given problem
    "n_episodes": 2000,
    "gamma": 0.99,
    "epsilon": 1.0,
    "epsilon_min": 0.05,

    # Config parameters: <- Display and Robustness testing
    "seed": 42,
    "show_progress": False,

    # Algorithm parameters: <- Parameters optimized for robust algorithms
    "alpha": 0.15,
    "epsilon_decay": 0.99,
}

In [ ]:
model, rewards, evals = run_experiment(alg_SARSA, params = SARSA_PARAMS)

print("\nFinal Summary:")
print(evals)

### Hyperparameter Optimization Pipeline

#### Q-learning

In [ ]:
main_log.info(f"Starting Optuna Hyperparameter Search: {alg_Q.__name__}")
best_trial = param_opt_pipeline(alg_Q)

best_trial.params

In [ ]:
fixed_best_trial = FixedTrial(best_trial.params)
final_params = get_params(fixed_best_trial, "alg_Q")

model, rewards, evals = run_experiment(alg_Q, "Q-learning")

print("\nFinal Summary:")
print(evals)

In [ ]:
mean_q, std_q = q_stats(model)
print(f"Q mean: {mean_q:.3f}, Q std: {std_q:.3f}")

In [ ]:
coverage = len(model.keys()) / np.prod(BINS)
coverage

#### n-step SARSA

In [ ]:
main_log.info(f"Starting Optuna Hyperparameter Search: {alg_nStep_SARSA.__name__}")
best_trial = param_opt_pipeline(alg_nStep_SARSA)

best_trial.params

In [ ]:
fixed_best_trial = FixedTrial(best_trial.params)
final_params = get_params(fixed_best_trial, "alg_nStep_SARSA")

model, rewards, evals = run_experiment(alg_nStep_SARSA, params = final_params)

print("\nFinal Summary:")
print(evals)

In [ ]:
mean_q, std_q = q_stats(model)
print(f"Q mean: {mean_q:.3f}, Q std: {std_q:.3f}")

In [ ]:
coverage = len(model.keys()) / np.prod(BINS)
coverage

#### REINFORCE

In [ ]:
main_log.info(f"Starting Optuna Hyperparameter Search: {alg_REINFORCE_B.__name__}")
best_trial = param_opt_pipeline(alg_REINFORCE_B)

best_trial.params

In [ ]:
fixed_best_trial = FixedTrial(best_trial.params)
final_params = get_params(fixed_best_trial, "alg_REINFORCE_B")

model, rewards, evals = run_experiment(alg_REINFORCE_B, params = final_params)

print("\nFinal Summary:")
print(evals)

In [ ]:
coverage = len(model.keys()) / np.prod(BINS)
coverage

#### SARSA($\lambda$)

In [ ]:
main_log.info(f"Starting Optuna Hyperparameter Search: {alg_SARSA_Lambda.__name__}")
best_trial = param_opt_pipeline(alg_SARSA_Lambda)

best_trial.params

In [ ]:
fixed_best_trial = FixedTrial(best_trial.params)
final_params = get_params(fixed_best_trial, "alg_SARSA_Lambda")

model, rewards, evals = run_experiment(alg_SARSA_Lambda, params = final_params)

print("\nFinal Summary:")
print(evals)

In [ ]:
mean_q, std_q = q_stats(model)
print(f"Q mean: {mean_q:.3f}, Q std: {std_q:.3f}")

In [ ]:
coverage = len(model.keys()) / np.prod(BINS)
coverage